# 3주차 텐서 기초 — 실습 6~10  (셀 1~17)

**목표**: 텐서를 NumPy 배열의 확장으로 이해하고(생성·인덱싱·shape·dtype·브로드캐스팅),
GPU 로 옮겨 **속도와 메모리 한계를 직접 측정**한다.

| 실습 | 내용 | 셀 | 교시 |
|---|---|---|---|
| 6 | 텐서 생성·속성·인덱싱·연결·산술·shape 변환·NumPy 다리 | 1~9 | 2교시 |
| 7 | 브로드캐스팅 | 10 | 2교시 |
| 8 | `.to("cuda")` 장치 이동 + 불일치 오류 | 11~13 | 3교시 |
| 9 | CPU vs GPU 행렬곱 속도 | 14~15 | 3교시 |
| 10 | 8GB 메모리 한계 체감 | 16~17 | 3교시 |

> **실행 전 확인** — 우측 상단 커널 이름이 **`Python (dl2026)`** 인지 보세요.
> 아니면 커널명을 눌러 `Python (dl2026)` 을 선택하세요. (2주차 실습 8에서 다룬 그것입니다.)

> **GPU 가 없어도 셀 1~17 은 오류 없이 실행됩니다.** 셀 11~17 의 GPU 부분(장치 오류 재현·GPU 속도·메모리)은 GPU 가 있을 때만 결과가 나오고,
> 없으면 안내 문구만 출력됩니다. **Colab 에서 실행하면 전부 가능합니다.**

---

## 실습 6 — 텐서 생성·인덱싱·shape 변환·dtype

In [7]:
# 셀 1 — 텐서 만들기
import torch
import numpy as np

a = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
print(a)
print("shape :", a.shape)        # NumPy 와 같다
print("dtype :", a.dtype)
print("차원   :", a.ndim)
# → tensor([[1, 2, 3],
# →         [4, 5, 6]])
# → shape : torch.Size([2, 3])
# → dtype : torch.int64
# → 차원   : 2


tensor([[1, 2, 3],
        [4, 5, 6]])
shape : torch.Size([2, 3])
dtype : torch.int64
차원   : 2


In [8]:
# 셀 2 — 여러 가지 생성 방법
print(torch.zeros(2, 3))                 # 0으로
print(torch.ones(2, 3))                  # 1로
print(torch.arange(12))                  # 0~11
print(torch.rand(2, 3))                  # 0~1 난수
print(torch.eye(3))                      # 단위행렬

# 모양을 변수에 담아 두고 쓰는 방식 — 실전 코드에서 자주 본다
shape = (2, 3)
print(torch.rand(shape).shape)           # torch.Size([2, 3])
print(torch.ones(shape).shape)           # torch.Size([2, 3])

x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

# ★ x 의 모양과 dtype 을 그대로 물려받는다
print(torch.ones_like(x))
# → tensor([[1, 1, 1],
# →         [1, 1, 1]])          ← dtype 도 int64 그대로

print(torch.rand_like(x, dtype=torch.float))
# → tensor([[0.3904, 0.6009, 0.2566],
# →         [0.7936, 0.9408, 0.1332]])

print(torch.zeros_like(x).shape)         # torch.Size([2, 3])


tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
tensor([[0.7075, 0.0444, 0.3698],
        [0.6342, 0.7124, 0.4483]])
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
torch.Size([2, 3])
torch.Size([2, 3])
tensor([[1, 1, 1],
        [1, 1, 1]])
tensor([[0.5738, 0.5010, 0.9877],
        [0.1153, 0.6597, 0.6787]])
torch.Size([2, 3])


**관찰 포인트** — `torch.zeros(2, 3)` 은 괄호 없이 `2, 3` 을 씁니다.
NumPy 는 `np.zeros((2,3))` 처럼 튜플이었죠. **이 정도가 차이의 전부**입니다.

In [9]:
# 셀 3 — 텐서를 만나면 찍어 보는 세 줄
t = torch.rand(3, 4)

print("모양   shape :", t.shape)     # 몇 행 몇 열인가
print("자료형 dtype :", t.dtype)     # 어떤 숫자로 담겨 있나
print("장치   device:", t.device)    # ★ 어디에 올라가 있나
# → 모양   shape : torch.Size([3, 4])
# → 자료형 dtype : torch.float32
# → 장치   device: cpu    ← ★ 지금 만든 텐서는 전부 CPU 에 있다

print(t.ndim, t.numel())             # 차원 수, 전체 원소 개수
# → 2 12


모양   shape : torch.Size([3, 4])
자료형 dtype : torch.float32
장치   device: cpu
2 12


**오류가 나면 이 세 줄부터 찍습니다.** `shape` · `dtype` · `device` — 학기 내내 쓰는 진단 습관입니다 (§4-2).


In [10]:
# 셀 4 — 자료형
i = torch.tensor([1, 2, 3])                       # 정수 → int64
f = torch.tensor([1.0, 2.0, 3.0])                 # 소수 → float32
print(i.dtype, f.dtype)
# → torch.int64 torch.float32

# 명시적으로 지정
x = torch.tensor([1, 2, 3], dtype=torch.float32)
print(x.dtype)

# 변환
print(i.float().dtype)      # int64 → float32
print(f.long().dtype)       # float32 → int64


torch.int64 torch.float32
torch.float32
torch.float32
torch.int64


| dtype | 크기 | 쓰임 |
|---|---|---|
| `torch.float32` | 4 byte | **딥러닝 기본** 대부분의 텐서 |
| `torch.float16` | 2 byte | 혼합정밀도 학습 (7주차 AMP) — 메모리 절반 |
| `torch.int64` | 8 byte | 정답 레이블, 인덱스 |
| `torch.bool` | 1 byte | 마스크 |

> 딥러닝 모델은 **`float32` 를 기대**합니다. 정수 텐서를 그냥 넣으면 오류가 납니다.
> `dtype` 은 **셀 16 의 메모리 계산**에서 다시 나옵니다 — 4 byte냐 2 byte냐가
> 8GB 안에 얼마나 담기는지를 결정합니다.

In [11]:
# 셀 5 — 인덱싱은 NumPy 와 완전히 같다
a = torch.arange(12).reshape(3, 4)
print(a)
# → tensor([[ 0,  1,  2,  3],
# →         [ 4,  5,  6,  7],
# →         [ 8,  9, 10, 11]])

print("a[0]      :", a[0])          # 첫 행
print("a[:, 1]   :", a[:, 1])       # 두 번째 열
print("a[1, 2]   :", a[1, 2])       # 특정 원소
print("a[a > 5]  :", a[a > 5])      # 조건 인덱싱 (불리언 마스크)
print("a[:2, 1:3]:", a[:2, 1:3])    # 슬라이싱

b = torch.ones(4, 4)
print(b[0])                     # tensor([1., 1., 1., 1.])

b[:, 1] = 0                     # ★ 두 번째 열 전체를 0 으로
print(b)
# → tensor([[1., 0., 1., 1.],
# →         [1., 0., 1., 1.],
# →         [1., 0., 1., 1.],
# →         [1., 0., 1., 1.]])

b[b == 0] = -1                  # ★ 조건에 맞는 자리만 바꾸기
print(b[0])                     # tensor([ 1., -1.,  1.,  1.])


tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
a[0]      : tensor([0, 1, 2, 3])
a[:, 1]   : tensor([1, 5, 9])
a[1, 2]   : tensor(6)
a[a > 5]  : tensor([ 6,  7,  8,  9, 10, 11])
a[:2, 1:3]: tensor([[1, 2],
        [5, 6]])
tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])
tensor([ 1., -1.,  1.,  1.])


In [12]:
# 셀 6 — 여러 텐서를 이어 붙인다
b = torch.ones(4, 4)
b[:, 1] = 0
print(b.shape)                        # torch.Size([4, 4])

t1 = torch.cat([b, b, b], dim=1)      # 옆으로 붙인다 (열 방향)
print(t1.shape)                       # torch.Size([4, 12])

t0 = torch.cat([b, b, b], dim=0)      # 아래로 쌓는다 (행 방향)
print(t0.shape)                       # torch.Size([12, 4])


torch.Size([4, 4])
torch.Size([4, 12])
torch.Size([12, 4])


**`dim` 이 헷갈릴 때** — `dim=n` 은 **n번째 차원이 늘어난다**는 뜻입니다. `dim=1` 이면 열이, `dim=0` 이면 행이 늘어납니다.


In [13]:
# 셀 7 — 요소별 곱 vs 행렬 곱
a = torch.tensor([[1., 2.],
                  [3., 4.]])

print(a * a)                # ① 요소별 곱 — 같은 자리끼리
# → tensor([[ 1.,  4.],        1×1=1   2×2=4
# →         [ 9., 16.]])       3×3=9   4×4=16

print(a @ a)                # ② 행렬 곱 — 행 × 열
# → tensor([[ 7., 10.],        1×1+2×3=7    1×2+2×4=10
# →         [15., 22.]])       3×1+4×3=15   3×2+4×4=22

print(a.mul(a).equal(a * a))          # True — mul() 이 * 와 같다
print(a.matmul(a.T).equal(a @ a.T))   # True — matmul() 이 @ 와 같다

# 원소가 하나뿐인 텐서 → 파이썬 숫자로 꺼내기
s = a.sum()
print(s)                          # tensor(10.)   ← 스칼라도 텐서다
print(s.item(), type(s.item()))
# → 10.0 <class 'float'>          ← 이제 파이썬 숫자

# 바꿔치기(in-place) — 이름이 밑줄 _ 로 끝나면 원본을 고친다
a.add_(5)                         # a = a + 5 를 a 자리에서
print(a)
# → tensor([[6., 7.],
# →         [8., 9.]])           ← a 자체가 바뀌었다


tensor([[ 1.,  4.],
        [ 9., 16.]])
tensor([[ 7., 10.],
        [15., 22.]])
True
True
tensor(10.)
10.0 <class 'float'>
tensor([[6., 7.],
        [8., 9.]])


**`.item()`** 은 4주차 학습 루프에서 매번 씁니다 — `total += loss.item()`.

**밑줄 `_`** 로 끝나는 연산은 원본을 덮어씁니다. 자동미분과 충돌할 수 있으니 **지금은 `_` 없는 쪽을 기본**으로 쓰세요 (§3-3).


In [14]:
# 셀 8 — 모양 바꾸기
a = torch.arange(12)
print(a.shape)                       # torch.Size([12])

b = a.reshape(3, 4)
print(b.shape)                       # torch.Size([3, 4])

c = a.reshape(2, -1)                 # -1 = 알아서 계산
print(c.shape)                       # torch.Size([2, 6])

# 차원 추가·제거 — 딥러닝에서 매우 자주 쓴다
d = b.unsqueeze(0)                   # 맨 앞에 차원 추가 (배치 차원)
print(d.shape)                       # torch.Size([1, 3, 4])
print(d.squeeze(0).shape)            # torch.Size([3, 4])

# 전치
print(b.T.shape)                     # torch.Size([4, 3])


torch.Size([12])
torch.Size([3, 4])
torch.Size([2, 6])
torch.Size([1, 3, 4])
torch.Size([3, 4])
torch.Size([4, 3])


> **`unsqueeze(0)`** 은 **배치 차원**을 만드는 데 씁니다.
> 모델은 항상 `(배치, ...)` 형태를 기대하는데, 이미지 한 장을 넣을 때
> `(3, 224, 224)` → `(1, 3, 224, 224)` 로 바꿔 줘야 합니다. **7주차부터 계속 씁니다.**

> **`reshape` vs `view`** — `view` 는 메모리가 연속일 때만 되고 아니면 오류입니다.
> **헷갈리면 `reshape`.** 거의 항상 잘 됩니다.

In [15]:
# 셀 9 — 한쪽을 고치면 다른 쪽도 바뀐다
# ① 텐서 → NumPy
t = torch.ones(5)
n = t.numpy()
print(t)                      # tensor([1., 1., 1., 1., 1.])
print(n)                      # [1. 1. 1. 1. 1.]

t.add_(1)                     # ★ 텐서만 고쳤는데
print(t)                      # tensor([2., 2., 2., 2., 2.])
print(n)                      # [2. 2. 2. 2. 2.]   ← NumPy 쪽도 같이 바뀐다

# ② NumPy → 텐서 — 반대 방향도 마찬가지
n2 = np.ones(5)
t2 = torch.from_numpy(n2)
np.add(n2, 1, out=n2)         # ★ NumPy 만 고쳤는데
print(t2)
# → tensor([2., 2., 2., 2., 2.], dtype=torch.float64)  ← 텐서도 바뀐다


tensor([1., 1., 1., 1., 1.])
[1. 1. 1. 1. 1.]
tensor([2., 2., 2., 2., 2.])
[2. 2. 2. 2. 2.]
tensor([2., 2., 2., 2., 2.], dtype=torch.float64)


**`from_numpy()` 와 `.numpy()` 는 복사가 아닙니다.** 두 객체가 같은 메모리를 봅니다.

한쪽을 밑줄 연산으로 고치면 다른 쪽도 바뀝니다. 사본이 필요하면 `.clone()` 을 쓰세요.

---

## 실습 7 — 브로드캐스팅

```
뒤쪽 차원부터 비교해서
  ① 같으면            → OK
  ② 둘 중 하나가 1이면 → 1인 쪽을 늘린다
  ③ 둘 다 아니면       → 오류
```

In [16]:
# 셀 10 — 브로드캐스팅
a = torch.arange(12).reshape(3, 4).float()
print("a :", a.shape)

# ① 스칼라 — 모든 원소에 적용
print((a + 100).shape)               # (3,4)

# ② (3,4) + (4,) — 각 행에 더해진다
row = torch.tensor([10., 20., 30., 40.])
print((a + row).shape)               # (3,4)
print(a + row)

# ③ (3,1) + (1,4) — 둘 다 늘어난다
col = torch.tensor([[1.], [2.], [3.]])   # (3,1)
r   = torch.tensor([[10., 20., 30., 40.]])  # (1,4)
print((col + r).shape)               # (3,4)

# ④ 실패 사례 — 일부러 오류를 내 본다
bad = torch.tensor([1., 2., 3.])     # (3,)
try:
    a + bad
except RuntimeError as e:
    print("오류 발생 →", e)
# → 오류 발생 → The size of tensor a (4) must match the size of tensor b (3) ...


a : torch.Size([3, 4])
torch.Size([3, 4])
torch.Size([3, 4])
tensor([[10., 21., 32., 43.],
        [14., 25., 36., 47.],
        [18., 29., 40., 51.]])
torch.Size([3, 4])
오류 발생 → The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 1


**④의 오류 메시지를 잘 읽어 두세요.**
`size of tensor a (4) must match ... b (3)` — **어느 차원이 안 맞는지 알려 줍니다.**
앞으로 학기 내내 만날 오류의 절반이 이 형태입니다.

**실전에서 쓰이는 곳**

| 상황 | 모양 |
|---|---|
| 배치 전체에 편향(bias) 더하기 | `(32, 256) + (256,)` |
| 이미지 정규화 (채널별 평균 빼기) | `(3, 224, 224) - (3, 1, 1)` |
| 배치의 각 샘플에 다른 가중치 | `(32, 10) * (32, 1)` |

---

## 실습 8 — `.to("cuda")` 장치 이동 + 장치 불일치 오류

여기부터 3교시입니다.

In [17]:
# 셀 11 — 텐서는 어디에 있는가
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"     # ★ 모든 코드의 첫머리
print("사용할 장치 :", device)

a = torch.rand(3, 4)
print("a.device :", a.device)          # → cpu

b = a.to(device)
print("b.device :", b.device)          # → cuda:0 (GPU 가 있으면)
print("a.device :", a.device)          # → cpu   ★ a 는 그대로다

# 처음부터 그 장치에 만들 수도 있다
c = torch.rand(3, 4, device=device)
print("c.device :", c.device)

# 다시 CPU로
d = b.to("cpu")
print("d.device :", d.device)


사용할 장치 : cpu
a.device : cpu
b.device : cpu
a.device : cpu
c.device : cpu
d.device : cpu


**관찰 포인트** — `a.device` 가 `.to()` 후에도 **여전히 `cpu`** 입니다.
**이동은 복사이고, 원본은 그대로 남습니다.**

> **함정** — `a.to("cuda")` 는 `a` 자신을 바꾸지 않습니다.
> 반드시 **결과를 받아야** 합니다. `a = a.to(device)` 또는 `b = a.to(device)`.
> (모델은 다릅니다 — `model.to(device)` 는 제자리에서 바뀝니다. 5주차에 다시 나옵니다.)

In [18]:
# 셀 12 — 장치를 섞으면 어떻게 되는가
cpu_t = torch.rand(3, 4)
gpu_t = torch.rand(3, 4, device=device)

if device == "cuda":
    try:
        result = cpu_t + gpu_t
    except RuntimeError as e:
        print("오류 발생 →", e)

    # 해결 — 같은 곳으로 모은다
    result = cpu_t.to(device) + gpu_t
    print("해결 후 :", result.device)
else:
    print("(CPU 환경이라 이 오류는 재현되지 않습니다)")
    print("GPU 환경에서는 다음 오류가 납니다:")
    print("  Expected all tensors to be on the same device,")
    print("  but found at least two devices, cuda:0 and cpu!")


(CPU 환경이라 이 오류는 재현되지 않습니다)
GPU 환경에서는 다음 오류가 납니다:
  Expected all tensors to be on the same device,
  but found at least two devices, cuda:0 and cpu!


**`Expected all tensors to be on the same device`** — 학기 내내 가장 자주 만날 오류 중 하나입니다.
메시지가 친절해서 `cuda:0 and cpu` 라고 **어느 둘이 섞였는지 알려 줍니다.**
해결은 항상 같습니다: **둘을 같은 device 로 모은다.**

> 모델은 GPU 에 올렸는데 데이터를 CPU 에서 만들어 넣는 실수가 가장 흔합니다.
> 5주차에 학습 루프를 짤 때 `xb.to(device)`, `yb.to(device)` 를
> 매 배치마다 쓰는 이유가 이것입니다.

In [19]:
# 셀 13 — GPU 텐서는 바로 numpy 로 못 간다
g = torch.rand(2, 3, device=device)

if device == "cuda":
    try:
        g.numpy()
    except TypeError as e:
        print("오류 발생 →", e)
    print(g.cpu().numpy())     # ← 해결: CPU로 먼저 옮긴다
else:
    print("(CPU 환경이라 바로 numpy 로 갑니다)")
    print(g.numpy())
    print("\nGPU 환경에서는 다음 오류가 납니다:")
    print("  can't convert cuda:0 device type tensor to numpy.")
    print("  Use Tensor.cpu() to copy the tensor to host memory first.")


(CPU 환경이라 바로 numpy 로 갑니다)
[[0.1933707  0.162085   0.7610509 ]
 [0.11857599 0.98501843 0.28599745]]

GPU 환경에서는 다음 오류가 납니다:
  can't convert cuda:0 device type tensor to numpy.
  Use Tensor.cpu() to copy the tensor to host memory first.


**오류 메시지가 해결 방법까지 알려 줍니다** (`Use Tensor.cpu()`).
PyTorch 오류는 대체로 친절합니다. **끝까지 읽는 습관**을 들이세요.

matplotlib 으로 그래프를 그릴 때도 `.cpu().numpy()` 가 필요합니다 — 6주차부터 계속 씁니다.

---

## 실습 9 — CPU vs GPU 행렬곱 속도 비교

> **측정할 때의 함정 — 동기화**
> GPU 연산은 **비동기**입니다. 파이썬이 `a @ b` 를 지나가도 GPU 는 아직 계산 중일 수 있습니다.
> 그냥 `time.time()` 으로 재면 *"엄청 빠르다"* 는 거짓 결과가 나옵니다.
> 해결: **`torch.cuda.synchronize()`** — GPU 가 끝날 때까지 기다린다.

In [ ]:
# 셀 14 — CPU vs GPU 행렬곱 속도
import time
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
N = 4096                                  # 4096 x 4096 행렬

def bench(dev, n=N, repeat=3):
    a = torch.rand(n, n, device=dev)
    b = torch.rand(n, n, device=dev)

    # 워밍업 — 첫 실행은 초기화 비용이 섞여 느리다
    for _ in range(2):
        _ = a @ b
    if dev == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(repeat):
        c = a @ b
    if dev == "cuda":
        torch.cuda.synchronize()          # ★ 반드시. 없으면 거짓 결과
    return (time.time() - t0) / repeat

cpu_t = bench("cpu")
print(f"CPU : {cpu_t*1000:8.1f} ms")

if device == "cuda":
    gpu_t = bench("cuda")
    print(f"GPU : {gpu_t*1000:8.1f} ms")
    print(f"→ GPU 가 약 {cpu_t/gpu_t:.0f} 배 빠름")
else:
    print("GPU 사용 불가 — Colab 에서 실행해 보세요")


**관찰 포인트** — 배수는 PC마다 크게 다릅니다. **수십 배~수백 배** 범위가 정상입니다.
중요한 건 정확한 숫자가 아니라 **자릿수가 다르다**는 감각입니다.

> **워밍업(warm-up)** 을 왜 하는지 주목하세요. 첫 번째 GPU 연산에는 커널 로딩·메모리 할당 같은
> 초기화 비용이 섞여 있습니다. 이걸 빼지 않으면 GPU 가 실제보다 느리게 측정됩니다.

In [ ]:
# 셀 15 — 크기에 따라 이득이 달라진다
for n in [64, 256, 1024, 4096]:
    c = bench("cpu", n)
    if device == "cuda":
        g = bench("cuda", n)
        print(f"N={n:5d}   CPU {c*1000:8.2f} ms   GPU {g*1000:7.2f} ms   →  {c/g:6.1f}배")
    else:
        print(f"N={n:5d}   CPU {c*1000:8.2f} ms")


**결과 해석** — **작은 연산에서는 GPU 가 오히려 느립니다.**
CPU→GPU 로 데이터를 보내고 결과를 받아 오는 **오버헤드**가 계산 시간보다 크기 때문입니다.
**GPU 는 "큰 일을 한꺼번에" 할 때만 이득**입니다.

> 이게 딥러닝에서 **배치(batch)** 를 쓰는 이유입니다.
> 이미지를 한 장씩 넣으면 GPU 가 놀고, 32장씩 묶어 넣으면 제대로 일합니다.
> **5주차 `DataLoader` 와 7주차 배치 크기 조정이 전부 여기서 나옵니다.**

> 이 출력은 **과제 제출물**입니다.

---

## 실습 10 — 8GB 메모리 한계 체감

```
텐서 메모리 = 요소 수 × dtype 바이트

float32 (4 byte) 기준
  (1000, 1000)    =   100만 개 × 4B  =    4 MB
  (4096, 4096)    =  1677만 개 × 4B  =   64 MB
  (10000, 10000)  =    1억 개 × 4B   =  400 MB
  (16384, 16384)  =  2.7억 개 × 4B   = 1024 MB = 1 GB

→ 8GB VRAM 에는 (16384, 16384) float32 텐서가 약 8개
```

**그런데 실제로는 8개를 못 올립니다. 왜일까요?**
PyTorch 자체가 쓰는 메모리와 연산 중간 결과가 자리를 차지하기 때문입니다.

In [ ]:
# 셀 16 — 메모리 사용량 관찰
import torch

def mem():
    a = torch.cuda.memory_allocated() / 1024**3      # 실제 텐서가 쓰는 양
    r = torch.cuda.memory_reserved()  / 1024**3      # PyTorch가 확보해 둔 양
    return f"allocated {a:5.2f} GB / reserved {r:5.2f} GB"

if device == "cuda":
    torch.cuda.empty_cache()
    print("시작       :", mem())

    x = torch.rand(16384, 16384, device="cuda")      # 1 GB
    print("1 GB 할당  :", mem())

    y = torch.rand(16384, 16384, device="cuda")      # +1 GB
    print("2 GB 할당  :", mem())

    del y                                            # 참조 제거
    print("y 해제 후  :", mem())                     # ← reserved 는 그대로

    torch.cuda.empty_cache()                         # 캐시 반환
    print("캐시 비움  :", mem())                     # ← 이제 줄어든다

    print(f"\n전체 VRAM  : {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")
    del x; torch.cuda.empty_cache()
else:
    print("(CPU 환경 — GPU 메모리 측정은 건너뜁니다)")
    print("Colab 에서 실행하면 다음과 같은 출력이 나옵니다:")
    print("  시작       : allocated  0.01 GB / reserved  0.02 GB")
    print("  1 GB 할당  : allocated  1.01 GB / reserved  1.02 GB")
    print("  2 GB 할당  : allocated  2.01 GB / reserved  2.02 GB")
    print("  y 해제 후  : allocated  1.01 GB / reserved  2.02 GB")
    print("  캐시 비움  : allocated  1.01 GB / reserved  1.02 GB")

**관찰 포인트** — `allocated` 와 `reserved` 가 다릅니다.
PyTorch 는 성능을 위해 **한 번 받은 메모리를 OS 에 바로 안 돌려주고 캐시**합니다.
`nvidia-smi` 에서 사용량이 안 줄어드는 것처럼 보이는 이유가 이것입니다.

In [ ]:
# 셀 17 — OOM 을 직접 만나 본다
if device == "cuda":
    torch.cuda.empty_cache()
    tensors = []
    i = 0
    try:
        for i in range(1, 20):
            tensors.append(torch.rand(16384, 16384, device="cuda"))   # 1 GB 씩
            print(f"{i} GB 할당 성공 — {mem()}")
    except torch.cuda.OutOfMemoryError:
        print(f"\n★ {i} GB 에서 OOM 발생. 8 GB 를 다 쓰지 못한다.")
        print("  이유: PyTorch 컨텍스트 + 단편화 + 연산 중간 버퍼")
    finally:
        tensors.clear()
        torch.cuda.empty_cache()
        print("정리 후 :", mem())
else:
    print("(CPU 환경 — OOM 실습은 Colab 에서 진행하세요)")
    print("8GB GPU 에서는 대개 6~7 GB 근처에서 OOM 이 납니다.")


**결과 해석** — 대개 **6~7GB 근처에서 OOM** 이 납니다. 8GB 를 다 못 씁니다.
학기 내내 만날 `CUDA out of memory` 오류의 정체가 이것입니다.

> **"GPU 는 빠르지만 좁다."** 이 감각이 이 과목 전체를 관통합니다.
> - **7주차** — 배치 크기를 줄이고 혼합정밀도(fp16)로 메모리 절반
> - **9주차** — LoRA 로 학습 파라미터를 1% 로, 4bit 양자화로 모델 크기 1/8
> - **14주차** — attention slicing 으로 확산 모델을 8GB 에 밀어 넣기
>
> 전부 **"좁은 GPU 에 어떻게 우겨넣을까"** 라는 하나의 질문에 대한 답입니다.

> **막히면**: OOM 이 난 뒤 계속 이상하면 **`커널 → 다시 시작`** 하세요.
> 파이썬 프로세스를 재시작하는 것이 가장 확실합니다.

> OOM 발생 지점(몇 GB)은 **과제 제출물**입니다.

---

### 과제 제출 전 확인

- [ ] 셀 1~17 이 실행되고 **출력이 저장된 상태**다 (`Ctrl+S`)
- [ ] 셀 10 ④의 오류 메시지를 보고 **어느 차원이 문제인지** 말할 수 있다
- [ ] 셀 15 의 표에서 **작은 N 에서 배수가 1 이하**인 것을 확인했다
- [ ] 셀 17 의 OOM 발생 지점을 메모했다 (GPU 가 있는 경우)
- [ ] `pip freeze > requirements.txt` 로 torch 를 포함해 갱신했다
- [ ] `python verify_week3.py` 로 자가 점검했다